# Reproducing Semenov et al. (2025), Figure 4, from dandiset 001273

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025). Excitation and polarization of
isolated neurons by high-frequency sine waves for temporal interference stimulation. *Cell Reports
Physical Science* 6, 102660. [doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660).
The claims file (`claims/001273/10.1016-j.xcrp.2025.102660.yaml`) was written from the NIH author
manuscript (PMC12320893) with its STAR Methods and supplement; this attempt read the same text.

Dandiset. [DANDI:001273](https://dandiarchive.org/dandiset/001273/0.250805.1356), version
0.250805.1356, "Kinetics of Membrane Charging and Its Dependence on Neuron Shape and Alignment with
the Electric Field". The paper cites the dandiset without a version. There are two published
versions, 0.250701.1624 and 0.250805.1356; they list the same 181 files with the same blobs, and
their metadata differ only in a second NIH award and project added to the later one. The most
recent published version is pinned, as the protocol asks when no version is cited.

What it holds. The pulsed laser strobe imaging of Figure 4: 181 NWB files, each one replicate
of 160 full-sensor images (512 x 512) of a FluoVolt-loaded neuron at 50-ns steps around a 2-us
pulse. It holds no ROIs, no debris ROI, no cell shape or orientation labels, and the electrode axis
for only some experiments (see the data check). The fast imaging of Figures 1 to 3, S1 and S2 and
the sine wave strobe imaging of Figures 5, 6 and S3 are in other dandisets (claims with
`data: elsewhere`); they are not tested in this notebook.

| Claim | Tested here |
|---|---|
| fig4-polarized-poles-face-electrodes | yes, on the experiments whose electrode axis is stated |
| fig4-orientation-time-constants | blocked: no shape or orientation labels |
| fig4-no-pole-difference | yes |
| fig4-discharging-slower | blocked: no shape or orientation labels (the pooled test is shown) |
| fig4-pooled-time-constants | yes |
| fig4-full-charge-within-3us | blocked: no labels for its reading (a) (reading (b) is shown) |
| fig3c-sustained-firing-2s | data elsewhere, tested in `elsewhere_001030.ipynb` |
| figs1-di8-anepps-control | data elsewhere, tested in `elsewhere_001272.ipynb` |
| fig1, fig2, fig2c, fig2d, fig3b, figs2, fig5, fig6, threshold-depolarization-estimate | blocked (data elsewhere, not tested; their dandisets are surveyed in `survey_other_dandisets.ipynb`) |

No random numbers are used, so no seed is needed.

In [ ]:
import time
import warnings
from collections import Counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import strobe

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = strobe.list_assets()
meta = strobe.read_all_metadata(assets)
print(f"{len(meta)} files read from dandiset {strobe.DANDISET_ID} version {strobe.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
files = pd.DataFrame(meta)
files["date"] = files.subject_id.str.split("_").str[1]
files["cell"] = files.subject_id.str.split("_").str[0]
print("Image shape and type:", Counter(files["shape"]), Counter(files.dtype))
print("Rate attribute (Hz):", Counter(files.rate_hz))
from datetime import datetime
unix_start = files.session_start.map(lambda s: datetime.fromisoformat(s).timestamp())
print("starting_time equals the session start as a Unix time (s):",
      bool(np.allclose(files.starting_time, unix_start)), "; range of starting_time:",
      files.starting_time.min(), "to", files.starting_time.max())
print("Species:", Counter(files.species), " age field:", Counter(files.age))
print("Distinct subject ids:", files.subject_id.nunique(), " recording dates:", sorted(files.date.unique()))
print("\nFiles per subject id, nominal amplitude and electrode axis:")
print(pd.crosstab([files.subject_id], [files.amplitude_v, files.electrode_axis.fillna("not given")]).to_string())
print("\nStepping direction:", Counter(files.direction))
print("Electrode text:", Counter(files.electrode_text.fillna("none")))
print("\nSeries description of the first file:\n", files.series_description.iloc[0])

In [ ]:
# Everything one file holds: every group and dataset with its shape, the scalar and text fields,
# and the attributes of the image series. ROIs, labels or notes would show up here.
def show(name, obj):
    if name.startswith("specifications"):
        return   # the NWB schema copies
    if isinstance(obj, h5py.Dataset):
        if obj.shape == () or (obj.dtype.kind in "OSU" and obj.size <= 4):
            v = obj[()]
            v = v.decode() if isinstance(v, bytes) else v
            print(f"  {name}: {str(v)[:160]}")
        else:
            print(f"  {name}: dataset {obj.shape} {obj.dtype}")
    else:
        print(f"  {name}/ (group)")
with strobe._open(meta[0]["url"]) as h:
    print(meta[0]["path"])
    h.visititems(show)
    print("Attributes of acquisition/StrobeImaging:",
          {k: (v.decode() if isinstance(v, bytes) else v) for k, v in h["acquisition/StrobeImaging"].attrs.items()})

The file holds one image series, the subject and the session fields, with no ROIs, processing
module or notes beyond the descriptions.

Every file holds 160 images of 512 x 512 pixels, as the Methods say (160 images over 8 us). The
Results' "1,440 images per experiment" (180 per replicate, to 6 us after the pulse) does not match;
the files have 160. The `rate` of 6 Hz is the camera rate, not the 50-ns step of the time axis,
which the files give only in the series description, and `starting_time` holds the session start
as a Unix time where NWB expects seconds after it. The subject ids are cells (cell number and
date), not animals; the age is given as P18D for neurons dissected at E18 and cultured for about a
week. The amplitude is the nominal pulse generator voltage (40 to 130 V); the description notes the
measured values (40 V gave 25 V, 80 V gave 62 V), and no file gives the field in V/cm. The paper
gives 140 V/cm for every experiment.

In [ ]:
groups = strobe.group_experiments(meta)
runs = pd.DataFrame([{
    "run": k, "subject_id": g[0]["subject_id"], "amplitude_v": g[0]["amplitude_v"],
    "electrode_axis": g[0]["electrode_axis"] or "not given", "files": len(g),
    "first": g[0]["session_start"][11:19], "last": g[-1]["session_start"][11:19],
    "directions": "".join(m["direction"][0].upper() for m in g)} for k, g in enumerate(groups)])
print(runs.to_string(index=False))
experiments = [g for g in groups if len(g) >= 7]
singles = [g for g in groups if len(g) < 7]
print(f"\nRuns of 7 or 8 replicates (experiments): {len(experiments)}; single files: {len(singles)} "
      f"({', '.join(g[0]['subject_id'] + ' at ' + str(g[0]['amplitude_v']) + ' V' for g in singles)})")

The files fall into runs of consecutive files of one cell at one amplitude and electrode axis. 21
runs have 8 files, stepped forward and backward in the pattern F B B F F B B F, and one (cell 1 of
January 17 at 40 V) has 7, the last forward file missing. These are taken as the experiments,
eight replicates each, as the Results describe. Six single files at 100 V, between runs on January 19,
are not replicated and are left out. That makes 22 experiments where the paper reports 19; nothing in the
files says which 19 the paper used (see the readings of the experiment set below). On January 19
each of cells 1 to 4 has two runs at 40 V, one with the electrodes "Up and Bottom" and one "Right
and Left": the two orthogonal field directions of the Methods. On January 17 each cell has a run
at 40 V and one at 80 V with the electrode position "not specified", and cell 6 of January 19 has
runs at 100 and 130 V on the same axis; the four March 11 cells have one run each. No file says
which electrode was the cathode, nor the cell's shape or orientation group of Figure 4D and 4E.

In [ ]:
# fig6-sine-less-than-pulse: are the 2-us, 257 V/cm pulse recordings of the nine cells of Figures 5
# and 6 in this dandiset? Compare its recording dates with the subjects of DANDI:001274 (asset paths
# only, at its most recent published version), and the electrode text of this dandiset.
import json, urllib.request
with urllib.request.urlopen(f"{strobe.API}/dandisets/001274/versions/0.250805.1401/assets/?page_size=1000") as f:
    paths_1274 = [a["path"] for a in json.load(f)["results"]]
dates_1274 = Counter(p.split("/")[0].split("-")[-1] for p in paths_1274)
print("DANDI:001274 files per recording date:", dict(dates_1274))
print("This dandiset, files per recording date:", dict(Counter(files.date)))
print("Shared dates:", sorted(set(dates_1274) & set(files.date)) or "none")
print("Files of this dandiset naming the three-electrode array (1.4 mm apart, 0.5 mm rods):",
      sum("Distance between electrodes 1.4 mm" in m["description"] for m in meta), "of", len(meta))

This dandiset's cells were recorded on January 17 and 19 and March 11, 2024, with the
three-electrode array of Figure 4 at nominal 40 to 130 V. DANDI:001274 holds cells recorded in May
and August 2024. The nine cells of Figures 5 and 6 (bipolar electrode, 257 V/cm) are therefore not
here; their pulse recordings would be in 001274, which this attempt did not test (attempt record).

In [ ]:
started = time.time()
results = []
for g in experiments:
    results.append(strobe.process_experiment(g))
print(f"{len(results)} experiments loaded and reduced in {time.time() - started:.0f} s")
exp = runs[runs.files >= 7].reset_index(drop=True)
exp["label"] = exp.subject_id + " " + exp.amplitude_v.astype(int).astype(str) + "V " + exp.electrode_axis.str[:4]
exp["dim_frames"] = [int(r["dim_frames"].sum()) for r in results]
exp["soma_px"] = [r["mask_px"] for r in results]
digests = [d for r in results for d in r["digests"]]
print("Distinct image arrays among", len(digests), "files:", len(set(digests)))
print("Distinct blobs among all", len(files), "files:", files.blob.nunique())
print("\nPixels in the soma and in each ROI (reading a, a clipped to the soma, b), per experiment:")
roi_sizes = pd.DataFrame([{"experiment": exp.label[k], "soma_px": r["mask_px"],
                           **{f"{a}_{p}": r["roi_px"][(a, p)] for a in ("a", "a_clip", "b") for p in ("dep", "hyp")}}
                          for k, r in enumerate(results)])
print(roi_sizes.to_string(index=False))

No two files hold the same images, so no recording is deposited twice.

Time order. Files recorded "in a backward manner" are taken to be stored in acquisition order,
which runs back in time; the helper reverses them. The check below compares, per experiment, the
polarization trace (dF of the depolarized minus the hyperpolarized ROI, reading (a), unscaled) of
the forward files with that of the backward files, reversed and as stored.

In [ ]:
rows = []
for k, (g, r) in enumerate(zip(experiments, results)):
    fwd = np.array([m["direction"] == "forward" for m in g])
    d = r["traces"][("a", "dep", None)]
    h = r["traces"][("a", "hyp", None)]
    pol = 100 * (d / np.nanmean(d[:, :20], 1, keepdims=True) - h / np.nanmean(h[:, :20], 1, keepdims=True))
    f, b = np.nanmean(pol[fwd], 0), np.nanmean(pol[~fwd], 0)
    ok = ~np.isnan(f) & ~np.isnan(b)
    ok_raw = ~np.isnan(f) & ~np.isnan(b[::-1])
    rows.append({"experiment": exp.label[k],
                 "r_reversed": np.corrcoef(f[ok], b[ok])[0, 1],
                 "r_as_stored": np.corrcoef(f[ok_raw], b[::-1][ok_raw])[0, 1],
                 "onset_frame": strobe.estimate_onset(np.nanmean(pol, 0))})
order = pd.DataFrame(rows)
print(order.round(3).to_string(index=False))
print("\nMedian correlation, backward files reversed:", round(order.r_reversed.median(), 3),
      "; as stored:", round(order.r_as_stored.median(), 3))
print("Pulse onset estimated from the data (frame), median:", round(order.onset_frame.median(), 2),
      " range:", order.onset_frame.min().round(2), "to", order.onset_frame.max().round(2),
      "; Methods: frame 20 (1 us after the first image)")
dim = np.concatenate([r["dim_frames"] for r in results])
print("\nDim frames dropped:", int(dim.sum()), "of", dim.size, "; files with a dim first acquired image:",
      int(sum(r["dim_frames"][i, 0 if m["direction"] == "forward" else -1]
              for g, r in zip(experiments, results) for i, m in enumerate(g))), "of", len(dim))

The backward files agree with the forward ones once reversed, and not as stored, so they are stored
in acquisition order and the helper's reversal is right. The pulse onset estimated from the
polarization traces is close to frame 20 in every experiment, as the Methods state ("starting 1 us
before the pulse"), so the fits use the Methods' timing: charging over frames 20 to 59, discharging
from frame 60. The estimated onsets range from 18 to 21 frames, and a shift of one frame changes a
charging tau noticeably, so the fits are repeated with each experiment's estimated onset as a
variant (pooled section). The first image acquired in most files is about half as bright as the others (a
failed flash); those frames, and the few other dim ones, are dropped.

Clock. The data hold one image per time step and no other stream, so there is no separate
clock to align; the time base is the frame index, and the check above shows that the stated pulse
timing agrees with the response.

The figure below shows, per experiment, the filtered CONTROL image with the soma outline and the
normalized RATIO image (depolarized red, hyperpolarized blue) with the extreme pixels of each sign
within the soma (+). The pixel counts of the soma and of every ROI are printed above.

In [ ]:
fig, axes = plt.subplots(4, 11, figsize=(22, 8.5))
for k, r in enumerate(results):
    a, b = axes[(k // 11) * 2, k % 11], axes[(k // 11) * 2 + 1, k % 11]
    a.imshow(np.log(r["control_small"]), cmap="gray")
    a.contour(r["mask_small"], [0.5], colors="y", linewidths=0.6)
    b.imshow(r["ratio_small"], cmap="RdBu_r", vmin=-0.12, vmax=0.12)
    b.contour(r["mask_small"], [0.5], colors="k", linewidths=0.4)
    for ax in (a, b):
        ax.plot(r["dep_px"][1] / 4, r["dep_px"][0] / 4, "r+", ms=8)
        ax.plot(r["hyp_px"][1] / 4, r["hyp_px"][0] / 4, "b+", ms=8)
        ax.set_xticks([]); ax.set_yticks([])
    a.set_title(exp.label[k], fontsize=7)
plt.tight_layout()
plt.show()
ratio_all = np.concatenate([r["ratio_small"].ravel() for r in results])
print("RATIO - 1 over all images, 0.1 and 99.9 percentiles:", np.percentile(ratio_all, [0.1, 99.9]).round(3),
      "; pixels outside the colour range -0.12 to 0.12:", int((np.abs(ratio_all) > 0.12).sum()), "of", ratio_all.size)
print("Extreme values within the soma (RATIO - 1), depolarized:",
      np.round([r["dep_extreme"] for r in results], 3))
print("hyperpolarized:", np.round([r["hyp_extreme"] for r in results], 3))

## Fits

Charging and discharging fits for every experiment, pole, ROI reading (O1 a and b), weighting
(O2, none or 1/SEM^2) and discharge amplitude (O3, free or fixed), on unscaled data (F2: the debris
ROI is not in the dataset). The scaling to the whole-frame mean is a variant, not a reading. A
fit that fails or ends at a bound of tau is dropped and counted.

In [ ]:
rows = []
for k, r in enumerate(results):
    for roi in "ab":
        for scale in (None, "frame"):
            for pole in ("dep", "hyp"):
                mean, sem = strobe.delta_f(r["traces"][(roi, pole, scale)])
                for weighted in (False, True):
                    for fixed in (False, True):
                        f = strobe.fit_experiment(mean, sem, strobe.NOMINAL_ONSET, weighted, fixed)
                        rows.append({"exp": k, "amplitude_v": exp.amplitude_v[k], "roi": roi,
                                     "scale": scale or "none", "pole": pole, "weighted": weighted,
                                     "fixed": fixed, **f})
fits = pd.DataFrame(rows)
fits["abs_dfmax"] = fits.dfmax.abs()
readings = ["roi", "weighted", "fixed"]
main = fits[fits.scale == "none"]
drops = main.groupby(readings + ["pole"]).agg(
    charge_fits=("tau_charge", "count"), discharge_fits=("tau_discharge", "count")).reset_index()
print(f"Fits per reading and pole, out of {len(results)} experiments (unscaled):")
print(drops.to_string(index=False))

Every experiment's mean trace and fits, for both ROI readings, unscaled, unweighted, free
discharge amplitude (cathodic red, anodic blue). A missing curve is a fit that failed or ended at
a bound.

In [ ]:
t = (np.arange(strobe.N_FRAMES) - strobe.NOMINAL_ONSET) * strobe.FRAME_US
for roi in ("a", "b"):
    fig, axes = plt.subplots(4, 6, figsize=(22, 12), sharex=True)
    lo, hi = np.inf, -np.inf
    for k, ax in enumerate(axes.flat):
        if k >= len(results):
            ax.axis("off"); continue
        for pole, colour in (("dep", "tab:red"), ("hyp", "tab:blue")):
            m, s = strobe.delta_f(results[k]["traces"][(roi, pole, None)])
            lo, hi = min(lo, np.nanmin(m)), max(hi, np.nanmax(m))
            ax.errorbar(t, m, s, fmt=".", ms=2, color=colour, alpha=0.4, lw=0.4)
            f = strobe.fit_experiment(m, s, strobe.NOMINAL_ONSET)
            tc, td = t[(t >= 0) & (t < 2)], t[t >= 2]
            ax.plot(tc, strobe.charging(tc, f["dfmax"], f["tau_charge"]), color=colour)
            ax.plot(td, f["dis_amp"] * np.exp(-(td - 2) / f["tau_discharge"]), color=colour)
        ax.axvline(0, ls=":", c="k"); ax.axvline(2, ls=":", c="k")
        ax.set_title(exp.label[k], fontsize=8)
    fig.suptitle(f"ROI reading ({roi}), dF (%) against time from the nominal onset (us)")
    plt.tight_layout()
    plt.show()
    print(f"Reading ({roi}): range of the plotted mean dF {lo:.1f} to {hi:.1f}% (each panel on its own axis)")

### Experiment sets

The paper reports 19 experiments; the dataset has 22. Two sets are run: (A) all 22, and (B) the 15
experiments at the nominal 40 V, the amplitude of most runs and of every run on January 19 and
March 11 but two; the paper gives one field, 140 V/cm, for all of Figure 4. Neither set is known to
be the paper's.

In [ ]:
sets = {"A: all 22": list(range(len(results))),
        "B: 40 V only": [k for k in range(len(results)) if exp.amplitude_v[k] == 40]}
print({name: len(ks) for name, ks in sets.items()})

## fig4-polarized-poles-face-electrodes

> A 2-us pulse polarizes mainly the membrane facing the electrodes, depolarized at the
> cathode-facing pole and hyperpolarized at the anode-facing one, while even large neurites show
> little or no polarization.

Criterion: per experiment, the filtered RATIO image of the Methods; the extreme depolarized and
hyperpolarized pixels of the cell lie on opposite sides of its centroid along the field axis, with
the angle between their separation and the field axis below 45 degrees, in more than half of the
experiments (of all experiments, as the shape labels are missing, O4). The field axis is taken
from the electrode text ("Up and Bottom" is the image's vertical axis, "Right and Left" its
horizontal axis; an assumption, as the files do not say how the image is oriented to the stage).
The January 17 runs have no electrode position and are left out. The polarity of the electrodes
is not given, so which pole faced the cathode cannot be checked; the test is on the axis.

In [ ]:
rows = []
for k, r in enumerate(results):
    axis = exp.electrode_axis[k]
    if axis == "not given":
        continue
    field = np.array([1.0, 0.0]) if axis == "vertical" else np.array([0.0, 1.0])   # (row, column)
    c = np.array(r["centroid"])
    dep, hyp = np.array(r["dep_px"], float), np.array(r["hyp_px"], float)
    sep = dep - hyp
    angle = np.degrees(np.arccos(abs(sep @ field) / np.linalg.norm(sep)))
    opposite = np.sign((dep - c) @ field) != np.sign((hyp - c) @ field)
    rows.append({"experiment": exp.label[k], "axis": axis, "dep_px": tuple(r["dep_px"]), "hyp_px": tuple(r["hyp_px"]),
                 "centroid": tuple(np.round(c).astype(int)), "separation_px": round(float(np.linalg.norm(sep)), 1),
                 "angle_deg": round(float(angle), 1), "opposite_sides": bool(opposite),
                 "passes": bool(opposite and angle < 45)})
poles = pd.DataFrame(rows)
print(poles.to_string(index=False))
print(f"\nPasses: {poles.passes.sum()} of {len(poles)} experiments with a stated axis "
      f"({100 * poles.passes.mean():.0f}%); median angle {poles.angle_deg.median():.1f} degrees")
for name, ks in sets.items():
    sub = poles[poles.experiment.isin(exp.label[ks])]
    print(f"Set {name}: {sub.passes.sum()} of {len(sub)}")
# Null reference: the same test with the axis swapped (the field direction the experiment did not use).
swapped = []
for k, r in enumerate(results):
    axis = exp.electrode_axis[k]
    if axis == "not given":
        continue
    field = np.array([0.0, 1.0]) if axis == "vertical" else np.array([1.0, 0.0])
    sep = np.array(r["dep_px"], float) - np.array(r["hyp_px"], float)
    swapped.append(np.degrees(np.arccos(abs(sep @ field) / np.linalg.norm(sep))) < 45)
print("With the axes swapped, angle below 45 degrees in", int(np.sum(swapped)), "of", len(swapped))

In [ ]:
# Variant: the soma mask opened with a disc of radius 8 pixels (largest part kept), which trims
# neurite stumps that the threshold mask can take in; extreme pixels and centroid recomputed.
yy, xx = np.mgrid[-8:9, -8:9]
disc = yy ** 2 + xx ** 2 <= 64
rows = []
for k, r in enumerate(results):
    axis = exp.electrode_axis[k]
    if axis == "not given":
        continue
    lab, n = strobe.ndimage.label(strobe.ndimage.binary_opening(r["mask"], disc))
    body = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    inside = np.where(body, r["ratio"], np.nan)
    dep = np.array(np.unravel_index(np.nanargmax(inside), inside.shape), float)
    hyp = np.array(np.unravel_index(np.nanargmin(inside), inside.shape), float)
    c = np.array(np.nonzero(body)).mean(axis=1)
    field = np.array([1.0, 0.0]) if axis == "vertical" else np.array([0.0, 1.0])
    sep = dep - hyp
    angle = np.degrees(np.arccos(abs(sep @ field) / np.linalg.norm(sep)))
    opposite = np.sign((dep - c) @ field) != np.sign((hyp - c) @ field)
    rows.append({"experiment": exp.label[k], "soma_px": r["mask_px"], "opened_px": int(body.sum()),
                 "dep_moved_px": round(float(np.linalg.norm(dep - np.array(r["dep_px"]))), 1),
                 "hyp_moved_px": round(float(np.linalg.norm(hyp - np.array(r["hyp_px"]))), 1),
                 "angle_deg": round(float(angle), 1), "passes": bool(opposite and angle < 45)})
opened = pd.DataFrame(rows)
print(opened.to_string(index=False))
print(f"Opened soma: passes in {opened.passes.sum()} of {len(opened)}; median angle {opened.angle_deg.median():.1f} degrees")

In [ ]:
# The neurite part is reported, not judged: RATIO - 1 within the soma and over the rest of the image.
rows = []
for k, r in enumerate(results):
    ratio, mask = r["ratio_small"], r["mask_small"]
    outside = ratio[~mask]
    rows.append({"experiment": exp.label[k], "soma_p99_abs": np.percentile(np.abs(ratio[mask]), 99),
                 "outside_p99_abs": np.percentile(np.abs(outside), 99),
                 "outside_px_beyond_half_soma_extreme": int((np.abs(outside) > 0.5 * max(r["dep_extreme"], -r["hyp_extreme"])).sum())})
neurites = pd.DataFrame(rows)
print(neurites.round(3).to_string(index=False))
print("Experiments where the 99th percentile outside the soma is below that within it:",
      int((neurites.outside_p99_abs < neurites.soma_p99_abs).sum()), "of", len(neurites))

**Result: reproduced, on the 14 experiments whose electrode axis is stated.** In 13 of 14 the
extreme depolarized and hyperpolarized soma pixels lie on opposite sides of the soma centroid with
their separation within 45 degrees of the field axis (median angle 13.8 degrees); the exception, 1_20240311,
is at 48.7 degrees. With the axes swapped, 1 of 14 passes. In the 40 V set, 10 of 11 pass. With the
soma mask opened to trim neurite stumps, 12 of 14 pass (median angle 17.1 degrees). The
outcome rests on the assumption that "Up and Bottom" is the image's vertical axis. The 8 January 17
runs cannot be tested, and the polarity of the electrodes is not given. The neurite part is not
judged; the 99th percentile of the absolute RATIO change outside the soma is below that within it
in 21 of 22 experiments (table above).

## fig4-pooled-time-constants

> Pooled over all 19 experiments, charging and discharging time constants averaged 0.7 and 0.91 us
> at the cathodic pole and 0.65 and 0.8 us at the anodic pole.

Criterion: the mean over the experiment fits; reproduced when all four are within the paper's SEMs
(0.08, 0.08, 0.05, 0.07 us), under every reading of O1 to O3 (F2 unscaled only, here).

In [ ]:
expected = {("dep", "tau_charge"): (0.7, 0.08), ("dep", "tau_discharge"): (0.91, 0.08),
            ("hyp", "tau_charge"): (0.65, 0.05), ("hyp", "tau_discharge"): (0.8, 0.07)}
def pooled(data, ks):
    rows = []
    for key, sub in data[data.exp.isin(ks)].groupby(["scale"] + readings):
        row = dict(zip(["scale"] + readings, key))
        within = True
        for (pole, q), (value, tol) in expected.items():
            v = sub[sub.pole == pole][q]
            name = ("cath" if pole == "dep" else "anod") + "_" + ("chg" if q == "tau_charge" else "dis")
            row[name] = v.mean(); row[name + "_sem"] = v.sem(); row[name + "_median"] = v.median(); row[name + "_n"] = v.count()
            within &= abs(v.mean() - value) <= tol
        row["all_within"] = within
        rows.append(row)
    return pd.DataFrame(rows)
for name, ks in sets.items():
    p = pooled(fits, ks)
    print(f"Set {name}, mean (SEM, median, n) of tau in us:")
    show_cols = ["scale"] + readings + [c for c in p.columns if c.endswith(("chg", "dis"))] + ["all_within"]
    print(p[show_cols].round(2).to_string(index=False))
    for c in ["cath_chg", "cath_dis", "anod_chg", "anod_dis"]:
        print("   ", c, "SEM range", p[c + "_sem"].min().round(2), "to", p[c + "_sem"].max().round(2),
              "; median range", p[c + "_median"].min().round(2), "to", p[c + "_median"].max().round(2))
    print()

In [ ]:
# Per-experiment time constants, main reading (a, unweighted, free), unscaled, to show the spread.
m = main[(main.roi == "a") & ~main.weighted & ~main.fixed].pivot(index="exp", columns="pole",
                                                                  values=["tau_charge", "tau_discharge", "dfmax"])
m.index = exp.label
print(m.round(2).to_string())
print("\nBy amplitude (median tau_charge, cathodic and anodic):")
print(main[(main.roi == "a") & ~main.weighted & ~main.fixed].groupby(["amplitude_v", "pole"]).tau_charge.median().round(2).to_string())

Variants (unscaled, both experiment sets), to show how much the outcome depends on choices
the verifier questioned:

- onset: the pulse onset estimated from each experiment's polarization trace (data check) in place
  of the nominal frame 20, for F0 and for both fits;
- window: fits whose tau exceeds the time it is fitted over (2 us for charging, 5 us for
  discharging) are dropped, as such values extrapolate beyond the data;
- a_clip: ROI reading (a) with the square clipped to the soma.

In [ ]:
def fit_all(roi, onsets):
    rows = []
    for k, r in enumerate(results):
        for pole in ("dep", "hyp"):
            on = onsets[k]
            mean, sem = strobe.delta_f(r["traces"][(roi, pole, None)], int(np.floor(on)))
            for weighted in (False, True):
                for fixed in (False, True):
                    f = strobe.fit_experiment(mean, sem, on, weighted, fixed)
                    late = np.nanmean(mean[strobe.NOMINAL_ONSET + strobe.PULSE_FRAMES - strobe.POLARIZED_FRAMES:
                                           strobe.NOMINAL_ONSET + strobe.PULSE_FRAMES])
                    rows.append({"exp": k, "roi": roi, "scale": "none", "pole": pole, "weighted": weighted,
                                 "fixed": fixed, "abs_late_dF": abs(late), **f})
    out = pd.DataFrame(rows)
    out["abs_dfmax"] = out.dfmax.abs()
    return out
nominal = [strobe.NOMINAL_ONSET] * len(results)
estimated = order.onset_frame.to_list()
window = main.copy()
window.loc[window.tau_charge > strobe.PULSE_FRAMES * strobe.FRAME_US, ["tau_charge", "dfmax", "abs_dfmax", "tau_discharge"]] = np.nan
window.loc[window.tau_discharge > (strobe.N_FRAMES - strobe.NOMINAL_ONSET - strobe.PULSE_FRAMES) * strobe.FRAME_US, "tau_discharge"] = np.nan
variants = {"onset": pd.concat([fit_all(roi, estimated) for roi in "ab"]),
            "window": window,
            "a_clip": fit_all("a_clip", nominal)}
for vname, data in variants.items():
    for name, ks in sets.items():
        p = pooled(data, ks)
        cols = [c for c in p.columns if c.endswith(("chg", "dis", "_median"))]
        print(f"Variant {vname}, set {name}: mean and median tau (us); all four means within the SEMs in",
              int(p.all_within.sum()), "of", len(p), "readings")
        print(p[readings + cols + ["all_within"]].round(2).to_string(index=False))
    print()

**Result: not reproduced.** Under no reading of O1 to O3 on unscaled data, in either experiment
set, are all four mean time constants within the paper's SEMs (`all_within` is False in every
row). In set A the cathodic discharging means are 1.37 to 4.20 us against 0.91 plus or minus 0.08,
and the anodic discharging means 1.32 to 2.22 us against 0.8 plus or minus 0.07; the charging means
(cathodic 0.65 to 1.01, anodic 0.62 to 2.28 us) come near the paper's under some readings. The
frame-scaled variant gives no row within tolerance either. Up to 6 of 22 fits per pole are dropped
(the counts are printed under Fits). The variants give the same: with the estimated onsets, with
fits whose tau exceeds its window dropped, and with ROI (a) clipped to the soma, no reading in
either set has all four means within the SEMs, and the set A cathodic discharging means stay at
1.43 to 3.77, 1.55 to 1.92 and 1.36 to 1.58 us. The ROIs are rebuilt and the debris scaling could
not be applied, so this may be a failure of the reconstruction.

## fig4-no-pole-difference

> The maximum optical TMP and the charging and discharging time constants did not differ between
> the cathode- and anode-facing poles, indicating no rectification of the field.

Criterion: two-sided paired t test, cathodic against anodic, of |dFmax|, charging tau and
discharging tau over the experiments; reproduced when none is significant at p < 0.05 under every
reading of the header; not reproduced when any is under every reading; blocked when they disagree.
Mean differences and 95% intervals are reported.

In [ ]:
rows = []
for name, ks in sets.items():
    for key, sub in fits[fits.exp.isin(ks)].groupby(["scale"] + readings):
        wide = sub.pivot(index="exp", columns="pole", values=["abs_dfmax", "tau_charge", "tau_discharge"])
        row = dict(zip(["scale"] + readings, key)); row["set"] = name
        for q in ["abs_dfmax", "tau_charge", "tau_discharge"]:
            pair = wide[q].dropna()
            d = pair["dep"] - pair["hyp"]
            res = stats.ttest_rel(pair["dep"], pair["hyp"])
            ci = stats.t.interval(0.95, len(d) - 1, d.mean(), d.sem())
            row[q + "_n"] = len(d); row[q + "_diff"] = d.mean(); row[q + "_ci"] = f"{ci[0]:.2f} to {ci[1]:.2f}"
            row[q + "_p"] = res.pvalue
        row["any_significant"] = any(row[q + "_p"] < 0.05 for q in ["abs_dfmax", "tau_charge", "tau_discharge"])
        rows.append(row)
poles_test = pd.DataFrame(rows)
cols = ["set", "scale"] + readings + [c for c in poles_test.columns if c.startswith(("abs", "tau"))] + ["any_significant"]
print(poles_test[cols].round(3).to_string(index=False))
unscaled = poles_test[poles_test.scale == "none"]
print("\nUnscaled readings with any significant difference:", int(unscaled.any_significant.sum()), "of", len(unscaled))
for q in ["abs_dfmax", "tau_charge", "tau_discharge"]:
    print(f"  {q}: p < 0.05 in {(unscaled[q + '_p'] < 0.05).sum()} of {len(unscaled)}")

In [ ]:
# Variants of the pole test: the onset, window and a_clip variants above, and |dF| over the last
# 0.5 us of the pulse (the POLARIZED window of the Methods) in place of the fitted |dFmax|, which
# needs no fit and so keeps every experiment.
def pole_tests(data, ks, quantities):
    rows = []
    for key, sub in data[data.exp.isin(ks)].groupby(readings):
        wide = sub.pivot(index="exp", columns="pole", values=quantities)
        row = dict(zip(readings, key))
        for q in quantities:
            pair = wide[q].dropna()
            d = pair["dep"] - pair["hyp"]
            ci = stats.t.interval(0.95, len(d) - 1, d.mean(), d.sem())
            row[q + "_n"] = len(d); row[q + "_diff"] = round(d.mean(), 3)
            row[q + "_ci"] = f"{ci[0]:.2f} to {ci[1]:.2f}"; row[q + "_p"] = stats.ttest_rel(pair["dep"], pair["hyp"]).pvalue
        rows.append(row)
    return pd.DataFrame(rows)
late = pd.concat([fit_all(roi, nominal) for roi in "ab"])
for vname, data, qs in [("|dF| over the last 0.5 us of the pulse", late, ["abs_late_dF"]),
                        ("onset", variants["onset"], ["abs_dfmax", "tau_charge", "tau_discharge"]),
                        ("window", variants["window"], ["abs_dfmax", "tau_charge", "tau_discharge"]),
                        ("a_clip", variants["a_clip"], ["abs_dfmax", "tau_charge", "tau_discharge"])]:
    for name, ks in sets.items():
        t_ = pole_tests(data, ks, qs)
        if vname.startswith("|dF|"):
            t_ = t_[~t_.weighted & ~t_.fixed]   # no fit, so the O2 and O3 readings do not apply
        n_sig = int((t_[[q + "_p" for q in qs]] < 0.05).any(axis=1).sum())
        print(f"Variant {vname}, set {name}: readings with p < 0.05 for any quantity: {n_sig} of {len(t_)}")
        print(t_.round(3).to_string(index=False))
    print()
# Where the |dF| difference comes from: cathodic minus anodic |dF| over the last 0.5 us, by nominal amplitude.
sub = late[~late.weighted & ~late.fixed].pivot_table(index=["roi", "exp"], columns="pole", values="abs_late_dF").reset_index()
sub["diff"] = sub["dep"] - sub["hyp"]
sub["amplitude_v"] = exp.amplitude_v[sub.exp].values
print("\nCathodic minus anodic |dF| over the last 0.5 us (%), by amplitude:")
print(sub.groupby(["roi", "amplitude_v"])["diff"].agg(["count", "mean", "min", "max"]).round(2).to_string())
for roi in "ab":
    s = sub[(sub.roi == roi) & (sub.amplitude_v <= 80)]
    print(f"Reading ({roi}), set A without the 100 and 130 V runs: n = {len(s)}, mean diff {s['diff'].mean():.2f}%, "
          f"p = {stats.ttest_1samp(s['diff'], 0).pvalue:.3f}")

**Result: reproduced.** No unscaled reading, in either experiment set, finds a difference between
the poles at p < 0.05 in |dFmax|, charging tau or discharging tau (0 of 16 readings), and none of
the frame-scaled variants does. The 95% intervals of the differences are wide (printed above), and
7 to 10 of 22 experiments lose a pair to a failed fit, so the test has little power. The onset and
a_clip variants find no difference either. Two variants do find one in |dF|, not in tau. With
fits whose tau exceeds its window dropped, |dFmax| differs at p = 0.046 in 2 of 8 set A readings
(n = 12). The |dF| over the last 0.5 us of the pulse, which needs no fit and keeps all 22
experiments, is larger at the anodic pole in set A (mean difference -2.86% for ROI a, p = 0.014;
-2.54% for ROI b, p = 0.011), but not in the 40 V set (p = 0.28 and 0.24). The difference grows
with amplitude (-0.95% at 40 V, -14.82% and -12.23% at 100 and 130 V for ROI a), and without the
100 and 130 V runs it is -1.79% (p = 0.048) and -1.92% (p = 0.040). The criterion's quantity,
the fitted |dFmax|, shows no difference under any of the header readings (O1 to O3); only the
window variant, which drops fits, gives p = 0.046 in 2 of its 8 set A readings. The outcome
follows the criterion's quantity and readings; the data as a whole leave a pole difference in
amplitude open in set A, in particular if the 100 and 130 V runs belong to the paper's 19.

## fig4-discharging-slower (blocked, shown in part)

> Discharging time constants were larger than charging ones in the same cell, except in elongated
> cells aligned with the field.

The exception needs the elongated parallel group, which the dataset does not label (O4), so the
claim is blocked. Reading (a) of the criterion, all experiments and both poles, is shown.

In [ ]:
rows = []
for name, ks in sets.items():
    for key, sub in main[main.exp.isin(ks)].groupby(readings):
        pair = sub[["tau_charge", "tau_discharge"]].dropna()
        res = stats.ttest_rel(pair.tau_discharge, pair.tau_charge)
        rows.append({"set": name, **dict(zip(readings, key)), "n_pole_fits": len(pair),
                     "mean_discharge_minus_charge": (pair.tau_discharge - pair.tau_charge).mean(),
                     "discharge_larger": int((pair.tau_discharge > pair.tau_charge).sum()), "p": res.pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: blocked (metadata insufficient).** Without the shape and orientation labels the
exception for elongated parallel cells cannot be tested. The pooled comparison of reading (a) is
printed above for each reading of the header.

## fig4-full-charge-within-3us (blocked, shown in part)

> Membranes exceed 95% of full charge in under 3 us, even in the largest neurons and with the
> least efficient alignment with the field.

Reading (a) needs the elongated parallel group (no labels). Reading (b), 3 times the largest
charging tau of any experiment below 3 us, is shown; if it held, (a) would hold too, since a group
mean cannot exceed the largest value.

In [ ]:
rows = []
for key, sub in main.groupby(readings):
    for pole in ("dep", "hyp"):
        v = sub[sub.pole == pole].tau_charge.dropna()
        rows.append({**dict(zip(readings, key)), "pole": pole, "n": len(v), "max_tau": v.max(),
                     "3x_max_tau": 3 * v.max(), "experiments_3tau_over_3us": int((3 * v > 3).sum())})
print(pd.DataFrame(rows).round(2).to_string(index=False))
# Variant: charging fits with tau above the 2-us pulse dropped (the window variant of the pooled section).
w = variants["window"]
for key, sub in w.groupby(readings):
    v = sub.tau_charge.dropna()
    print(dict(zip(readings, key)), f"n = {len(v)}, max tau {v.max():.2f} us, pole fits with 3 tau over 3 us: {int((3 * v > 3).sum())}")

**Result: blocked (metadata insufficient).** Reading (b) fails under every reading of the header
(3 to 5 experiments per pole have 3 tau above 3 us), so it cannot stand in for reading (a), which
needs the group labels. It fails too with charging fits whose tau exceeds the 2-us pulse dropped
(the largest tau is then 1.64 to 1.97 us, and 4 to 6 pole fits have 3 tau above 3 us).

## fig4-orientation-time-constants (blocked)

The groups of Figure 4D and 4E (elongated parallel, elongated perpendicular, random) are not in the
dataset, and the paper judged them by eye (O4), so the claim is blocked. For reference, the
charging tau of the four January 19 cells imaged on both axes is printed (main reading); which
axis was parallel to each cell's long axis is not known.

In [ ]:
sub = main[(main.roi == "a") & ~main.weighted & ~main.fixed].copy()
sub["cell"] = exp.subject_id[sub.exp].values
sub["axis"] = exp.electrode_axis[sub.exp].values
both = sub[sub.cell.isin(["1_20240119", "2_20240119", "3_20240119", "4_20240119"])]
print(both.pivot_table(index=["cell", "pole"], columns="axis", values="tau_charge").round(2).to_string())

**Result: blocked (metadata insufficient).**

## Summary

| Claim | Outcome |
|---|---|
| fig4-polarized-poles-face-electrodes | reproduced (13 of the 14 experiments with a stated axis; assumed image orientation) |
| fig4-orientation-time-constants | blocked, metadata insufficient |
| fig4-no-pole-difference | reproduced |
| fig4-discharging-slower | blocked, metadata insufficient |
| fig4-pooled-time-constants | not reproduced |
| fig4-full-charge-within-3us | blocked, metadata insufficient |
| fig3c-sustained-firing-2s | partially reproduced (`elsewhere_001030.ipynb`) |
| figs1-di8-anepps-control | not reproduced (`elsewhere_001272.ipynb`) |
| other claims on other dandisets | blocked (attempt record) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the ROIs, including the debris ROI used to scale out
laser-flash fluctuations, are not in the dataset; the cell and polarized ROIs are rebuilt from the
RATIO image as the Methods describe, and the traces are unscaled. The paper's 19 experiments
cannot be identified among the 22 runs of the dataset. Fits are least squares in SciPy, not
Origin 8.0. No debris ROI was attempted: the paper chose it by eye on stained debris, and the
bright spots in the RATIO images polarize too, so any automatic choice would stand in for the
paper's no better than the whole-frame scaling shown as a variant.

What was found about the dataset: no ROIs, shape or orientation labels; the electrode axis is
missing for the January 17 runs and the polarity for all; files recorded backward are stored in
reverse time with nothing in the file to say so beyond the description; the time base (50 ns per
frame) is only in prose, with a 6 Hz rate and a Unix `starting_time`; 22 runs against the paper's
19 experiments, plus six single files; 160 images per replicate against the Results' 180. The
nine cells of Figures 5 and 6 are not in this dandiset (no recording date in common with
DANDI:001274, and every file here names the three-electrode array of Figure 4).